In [1]:
##########
# SETUP
##########

import sys, os
sys.path.append(os.path.abspath('..'))
sys.path.append('.')
from utils import loadFSL, mkdir_no_exist, FSLeyesServer

import module as lmod
await lmod.purge(force=True)
await lmod.load('fsl/6.0.7.18')
loadFSL()

from fsl.wrappers import bet, fast, flirt
import nibabel as nib
import numpy as np
import pandas as pd
from nilearn import plotting
import os.path as op
import subprocess
import glob

In [2]:
########
# PATHS
########

base = os.path.abspath("")
t1_path     = os.path.join(base, 'T1', 'T1w.nii.gz')
func_path   = os.path.join(base, 'fMRI', 'tfMRI_LANGUAGE_LR', 'tfMRI_LANGUAGE_LR.nii.gz')
events_path = os.path.join(base, 'fMRI', 'tfMRI_LANGUAGE_LR', 'tfMRI_LANGUAGE_LR_task_onsets.csv')

out_dir = os.path.join(base, 'derivatives')
mkdir_no_exist(out_dir)

In [3]:
%gui wx


(ipykernel_launcher.py:6887): dbind-WARNING **: 14:38:49.518: Couldn't connect to accessibility bus: Failed to connect to socket /run/user/1333487151/at-spi/bus_100: No such file or directory


In [4]:
fsleyesDisplay = FSLeyesServer()
fsleyesDisplay.show()

14:38:55: Debug: Adding duplicate image handler for 'Windows bitmap file'
14:38:55: Debug: Adding duplicate animation handler for '1' type
14:38:55: Debug: Adding duplicate animation handler for '2' type
14:38:55: Debug: Adding duplicate image handler for 'Windows bitmap file'
14:38:55: Debug: Adding duplicate animation handler for '1' type
14:38:55: Debug: Adding duplicate animation handler for '2' type

(ipykernel_launcher.py:6887): Gtk-CRITICAL **: 14:38:56.107: gtk_window_resize: assertion 'width > 0' failed

(ipykernel_launcher.py:6887): Gtk-CRITICAL **: 14:38:57.981: gtk_window_resize: assertion 'width > 0' failed


In [5]:
# Looking at the T1 image

fsleyesDisplay.load(t1_path)

In [5]:
#################
# Skull Stripping
#################

t1_path_strip = op.join(out_dir, 'T1w_brain')
subprocess.run(['bet', t1_path, t1_path_strip, '-m', '-R'])

In [6]:
# Looking at stripped brain

fsleyesDisplay.load(t1_path_strip)

In [7]:
##############
# Segmentation
##############

t1_path_seg = op.join(out_dir, 'T1w_fast')
fast_target = t1_path_strip
fast(imgs=[fast_target], out=t1_path_seg, n_classes = 3)

{}

In [9]:
######################
# Display Segmentation
######################
t1_path_seg = op.join(out_dir, 'T1w_fast')

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(t1_path_strip)
fsleyesDisplay.load(glob.glob(t1_path_seg + '*pve_0*')[0])
fsleyesDisplay.load(glob.glob(t1_path_seg + '*pve_1*')[0])
fsleyesDisplay.load(glob.glob(t1_path_seg + '*pve_2*')[0])
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[1]).cmap = 'Red'
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[2]).cmap = 'Green'
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[3]).cmap = 'Blue'

In [21]:
###############
# Registration
###############

mni_template = op.expandvars(op.join('$FSLDIR', 'data', 'standard', 'MNI152_T1_1mm_brain'))

source = t1_path_strip # Fill me
reference = mni_template # Fill me
result = op.join(out_dir, 'T1w_mni')
flirt(source, reference, out=result)


Final result: 
1.078909 -0.015680 0.008415 -6.129257 
0.023288 1.123096 -0.006228 -16.343431 
-0.020852 -0.011642 1.191129 -8.805752 
0.000000 0.000000 0.000000 1.000000 



{}

In [22]:
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(reference) 
fsleyesDisplay.load(result)

In [23]:
costs_to_consider = [ 'leastsq', 'normcorr' ] # fill me with the relevant costs

for c in costs_to_consider:
    flirt(source, reference, out=result + '_' + c, cost=c)


Final result: 
1.165288 -0.025935 0.016915 -13.515540 
0.029091 1.222016 -0.028761 -25.740137 
-0.015456 -0.024104 1.452098 -29.121278 
0.000000 0.000000 0.000000 1.000000 


Final result: 
1.070191 -0.015001 0.009304 -5.499236 
0.023592 1.115159 -0.005633 -15.657697 
-0.020778 -0.010821 1.178435 -7.862419 
0.000000 0.000000 0.000000 1.000000 



In [24]:
for c in costs_to_consider:
    fsleyesDisplay.load(result + '_' + c)